# 03 — Cohort Retention Analysis

**Olist Customer Intelligence Platform**

Group customers by the month of their **first** order (their acquisition cohort),
then track what share of each cohort places another order in each subsequent
month. Rendered as a heatmap, this is the clearest possible view of the
retention problem.

> **Reading the result:** month 0 is always 100% (everyone ordered in their own
> acquisition month). With Olist's ~97% one-time-buyer base, expect a **cliff** —
> retention collapses to low single digits by month 1. That collapse is the
> headline, not a bug.

In [1]:
import duckdb
import pandas as pd
import plotly.express as px
from pathlib import Path

DB_PATH = None
for candidate in (Path("../data/olist.duckdb"), Path("data/olist.duckdb")):
    if candidate.exists():
        DB_PATH = candidate.resolve()
        break
if DB_PATH is None:
    raise FileNotFoundError("olist.duckdb not found — run scripts/load_raw.py and `dbt build`.")

con = duckdb.connect(str(DB_PATH), read_only=True)


def q(sql: str) -> pd.DataFrame:
    """Run SQL against the Olist DuckDB and return a DataFrame."""
    return con.execute(sql).df()


print(f"Connected (read-only) to {DB_PATH.name}")

Connected (read-only) to olist.duckdb


## Step 1 — Build the cohort table (your SQL)

Write one query that returns a **tidy/long** table with exactly these columns:

| column | meaning |
|---|---|
| `cohort_month` | the month of the customer's first delivered order (a date, truncated to month) |
| `month_index` | whole months between the order month and the cohort month (0, 1, 2, …) |
| `cohort_size` | number of customers in that cohort (same for every row of a cohort) |
| `active_customers` | distinct customers from that cohort who ordered in that `month_index` |
| `retention_pct` | `100.0 * active_customers / cohort_size` |

**Suggested shape (CTEs):**
1. `first_orders` — each `customer_unique_id`'s first delivered order month
   (`date_trunc('month', min(order_purchase_timestamp))`). Use delivered orders only.
2. `orders_indexed` — join every delivered order back to its customer's cohort
   month; compute `month_index = date_diff('month', cohort_month, order_month)`.
3. `sizes` — `count(distinct customer_unique_id)` per `cohort_month`.
4. Final — group by `cohort_month, month_index`, count distinct active customers,
   join `sizes`, compute `retention_pct`. Assign the result to a variable named
   `cohort`.

⚠️ Reminders: `customer_unique_id` (not `customer_id`); delivered orders only, for
consistency with everything else.

In [3]:
cohort = q("""
with first_orders as (
    select
        customer_unique_id,
        date_trunc('month', min(order_purchase_timestamp)) as cohort_month
    from mart_orders
    where order_status = 'delivered'
    group by 1
),

orders_indexed as (
    select
        f.customer_unique_id,
        f.cohort_month,
        date_diff(
            'month',
            f.cohort_month,
            date_trunc('month', o.order_purchase_timestamp)
        ) as month_index
    from first_orders f
    join mart_orders o
        on f.customer_unique_id = o.customer_unique_id
        and o.order_status = 'delivered'
),

sizes as (
    select
        cohort_month,
        count(distinct customer_unique_id) as cohort_size
    from first_orders
    group by 1
),

cohort_activity as (
    select
        cohort_month,
        month_index,
        count(distinct customer_unique_id) as active_customers
    from orders_indexed
    group by 1, 2
)

select
    a.cohort_month,
    a.month_index,
    s.cohort_size,
    a.active_customers,
    round(100.0 * a.active_customers / s.cohort_size, 2) as retention_pct
from cohort_activity a
join sizes s
    on a.cohort_month = s.cohort_month
order by a.cohort_month, a.month_index
""")
cohort.head()

,cohort_month,month_index,cohort_size,active_customers,retention_pct
0,2016-09-01,0,1,1,100.00
1,2016-10-01,0,262,262,100.00
2,2016-10-01,6,262,1,0.38
3,2016-10-01,9,262,1,0.38
4,2016-10-01,11,262,1,0.38


## Step 2 — Render the heatmap (provided)

This pivots your long table into the cohort grid. We plot `month_index >= 1`
only — month 0 is trivially 100% and would wash out the color scale, hiding the
faint but real returning-customer signal.

In [4]:
plot_df = cohort[cohort["month_index"] >= 1].copy()
plot_df["cohort_month"] = plot_df["cohort_month"].astype(str).str.slice(0, 7)

pivot = plot_df.pivot(index="cohort_month",
                      columns="month_index",
                      values="retention_pct").sort_index()

fig = px.imshow(
    pivot,
    labels=dict(x="Months since first order", y="Acquisition cohort",
                color="Retention %"),
    color_continuous_scale="Blues",
    aspect="auto",
    text_auto=".1f",
    title="Cohort retention — % of each cohort ordering again N months later",
)
fig.update_xaxes(side="top")
fig.show()

## Step 3 — What the heatmap shows

**Takeaway:** _Retention cliffs immediately after month 0: weighted month-1 retention is ~0.5%, and it stays below 1% thereafter. A few 2017 cohorts (Sep–Nov) retain slightly better (~0.6–0.7%), but the pattern is uniformly weak. The bottom-right of the grid is empty because recent cohorts haven't had enough elapsed months yet (right-censoring / the cohort triangle). This proves ~97% of customers are one-and-done — invest in a 30–60 day win-back program, not long-horizon loyalty perks._